# Task 5: Auto-Tagging Support Tickets Using an LLM

**DevelopersHub Corporation — AI/ML Engineering Internship (Advanced Task Set)**

**Author:** _[Your Name Here]_
**Date:** _[Submission Date]_

---

This notebook builds a system that automatically tags free-text customer support
tickets with the most relevant category labels using a Large Language Model (LLM),
via two complementary strategies:

1. **Zero-shot classification** — the model is given the ticket text and a list of
   candidate tags, with no labeled examples, and must infer the most likely tags
   purely from its pretrained language understanding.
2. **Few-shot prompting** — the model is given a small number of labeled example
   tickets directly in the prompt (in-context learning) before being asked to tag
   a new ticket, which typically improves accuracy and consistency of the output
   format.

For every ticket, the pipeline outputs the **top 3 most probable tags**, and the
notebook quantitatively compares zero-shot vs. few-shot performance.


## 1. Problem Statement & Objective

### Problem
Support teams receive a high volume of free-text tickets every day (bug reports,
billing questions, feature requests, account issues, etc.). Manually reading and
tagging every ticket before routing it to the right team is slow, inconsistent
across agents, and does not scale as ticket volume grows.

### Objective
Build an automated ticket-tagging system that:

- Uses a **pretrained LLM** to classify tickets into predefined support categories
  **without requiring a large labeled training set** (zero-shot), and
- Improves on that baseline using **few-shot prompting**, where a handful of
  labeled examples are shown to the model as in-context guidance.
- Compares the two approaches quantitatively using standard classification metrics.
- Returns the **top 3 most probable tags per ticket**, since real tickets are
  often multi-faceted (e.g. a ticket can plausibly be both a "Bug Report" and a
  "Technical Issue"), and giving agents a ranked shortlist is more useful than a
  single forced label.

### Tag Taxonomy
The system classifies each ticket into one of 8 categories:

| Tag | Description |
|---|---|
| Technical Issue | App/website performance, crashes, connectivity problems |
| Billing & Payment | Charges, invoices, refund amounts, payment methods |
| Account Access | Login, password reset, 2FA, account lockouts |
| Feature Request | Suggestions for new functionality |
| Bug Report | Unexpected/incorrect product behavior (not general performance) |
| Cancellation & Refund | Subscription cancellation, downgrades, refund requests |
| Product Complaint | Dissatisfaction with product quality, UX, or service |
| General Inquiry | Informational questions (pricing, hours, policies, integrations) |

### Skills Demonstrated
Prompt engineering · LLM-based text classification · Zero-shot & few-shot learning ·
Multi-class prediction & ranking · Evaluation methodology


## 2. Environment Setup

This notebook uses:
- `transformers` (Hugging Face) — for the zero-shot classification model
  (`facebook/bart-large-mnli`) and the few-shot generative model
  (`google/flan-t5-base`).
- `scikit-learn` — for evaluation metrics (accuracy, precision, recall, F1).
- `pandas`, `matplotlib`, `seaborn` — for data handling and visualization.

> **Note:** The first run will download the pretrained model weights
> (a few hundred MB) from the Hugging Face Hub, so an internet connection is
> required the first time each model is used. This notebook was designed to run
> end-to-end on a free Google Colab CPU runtime (GPU speeds things up but is not
> required).


In [ ]:
# This installs everything needed on a fresh Google Colab runtime.
# torch is already preinstalled on Colab; the rest are not, so we install them here.
!pip install -q transformers sentencepiece accelerate scikit-learn pandas matplotlib seaborn

import warnings
warnings.filterwarnings("ignore")

import re
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

random.seed(42)
np.random.seed(42)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

print("Libraries imported successfully.")


## 3. Dataset Loading & Preprocessing

**Dataset:** a curated set of **96 free-text support tickets**, evenly balanced
across the 8 tag categories above (12 tickets per category). Each row has:

- `ticket_text` — the raw customer message
- `true_tag` — the ground-truth category, used only for evaluation

A synthetic-but-realistic dataset is hand-authored and generated directly in the
cell below (no file upload needed — this notebook is fully self-contained and
runs end-to-end in Colab with nothing but the notebook itself), because the task
brief does not point to one fixed, license-clear source dataset.

> **To use your own real ticket data instead:** skip the generation cell below
> and instead run `df = pd.read_csv("your_file.csv")`, where your CSV has the
> same two columns (`ticket_text`, `true_tag`) — every cell after this one will
> work unchanged.


In [ ]:
import csv

CATEGORIES = {
    "Technical Issue": [
        "The app keeps crashing every time I try to open the dashboard on my Android phone.",
        "Your website is extremely slow today, pages take over 30 seconds to load.",
        "I can't upload files larger than 5MB, the upload just fails silently.",
        "The mobile app freezes on the loading screen after the last update.",
        "Video calls keep dropping every few minutes during meetings.",
        "The search bar returns no results even for items I know exist in my account.",
        "Push notifications stopped working since yesterday's update.",
        "The desktop client won't launch, it just shows a blank white screen.",
        "Syncing between my phone and laptop app has stopped working entirely.",
        "I keep getting a 500 internal server error when I try to save my settings.",
        "The PDF export feature produces corrupted files that won't open.",
        "The app is draining my battery extremely fast even when running in the background.",
    ],
    "Billing & Payment": [
        "I was charged twice for my monthly subscription this billing cycle.",
        "My credit card was declined even though it has sufficient funds available.",
        "Can you explain why my invoice this month is higher than usual?",
        "I need a copy of my receipt from last month's payment for tax purposes.",
        "The discount code I applied at checkout was not reflected in my final bill.",
        "I want to update the credit card on file for my recurring subscription.",
        "There is an unrecognized charge on my statement from your company.",
        "My annual plan renewed early and I was billed before the expected date.",
        "I was charged in the wrong currency during checkout, please correct this.",
        "Please send me an itemized breakdown of all charges for the past quarter.",
        "I upgraded my plan but I'm still being billed at the old, lower rate.",
        "Is it possible to switch from monthly billing to annual billing?",
    ],
    "Account Access": [
        "I forgot my password and the reset link in the email is not working.",
        "I'm locked out of my account after entering the wrong password too many times.",
        "Two-factor authentication codes are not arriving on my registered phone number.",
        "I can no longer log in even though I'm sure my credentials are correct.",
        "My account was suspended and I don't understand why.",
        "I need to change the email address associated with my account.",
        "Someone else seems to have accessed my account without permission.",
        "I lost access to the authenticator app I use for two-factor login.",
        "How do I transfer ownership of this account to a new team administrator?",
        "My session keeps logging me out every few minutes for no reason.",
        "I never received the account verification email after signing up.",
        "Please help me merge two separate accounts I accidentally created.",
    ],
    "Feature Request": [
        "It would be great if you could add dark mode to the mobile app.",
        "Could you add support for exporting reports directly to Google Sheets?",
        "Please consider adding a bulk-edit option for managing multiple items at once.",
        "I'd love to see keyboard shortcuts added for the most common actions.",
        "Can you add an offline mode so I can keep working without internet access?",
        "It would help a lot to have calendar integration with Outlook.",
        "Please add the ability to customize notification preferences per project.",
        "A public API would be incredibly useful for automating our workflows.",
        "Could you add multi-language support for non-English speaking users?",
        "I'd like a way to schedule recurring tasks automatically.",
        "Please consider adding single sign-on (SSO) support for enterprise accounts.",
        "It would be nice to have a drag-and-drop file organizer in the dashboard.",
    ],
    "Bug Report": [
        "When I click 'Save Draft', the page reloads and my changes are lost.",
        "The date picker shows the wrong month when selecting dates near year-end.",
        "Filtering by tag returns items that don't actually have that tag applied.",
        "The total shown in my cart doesn't match the sum of the individual items.",
        "Editing a shared document sometimes duplicates the entire page.",
        "The chart on the analytics page displays negative values that shouldn't exist.",
        "Deleting one item accidentally deletes an unrelated item as well.",
        "The timezone displayed in scheduled events is off by several hours.",
        "Sorting by 'newest first' actually sorts items oldest first.",
        "The progress bar gets stuck at 99% and never completes the upload.",
        "Comments I post sometimes appear twice on the same thread.",
        "The currency symbol displayed is incorrect for accounts outside the US.",
    ],
    "Cancellation & Refund": [
        "I would like to cancel my subscription effective immediately.",
        "Please process a refund for the annual plan I purchased by mistake.",
        "How do I downgrade from the premium plan to the free tier?",
        "I want to cancel my trial before it converts into a paid subscription.",
        "Can I get a prorated refund for the unused months on my current plan?",
        "I accidentally purchased the wrong add-on and need a refund.",
        "Please cancel my account and delete all associated data permanently.",
        "I was charged after cancelling, please refund this payment.",
        "What is your refund policy if I'm not satisfied within the first 30 days?",
        "I'd like to pause my subscription for two months instead of cancelling outright.",
        "Please confirm that my cancellation request has been fully processed.",
        "I need to cancel one seat on our team plan but keep the rest active.",
    ],
    "Product Complaint": [
        "The quality of customer support has really declined over the past few months.",
        "I'm very disappointed that a promised feature was removed without warning.",
        "The new interface is confusing and much harder to use than the old one.",
        "Your product stopped supporting a device I rely on every day for work.",
        "I feel misled by the marketing, the product doesn't do what was advertised.",
        "The onboarding process was frustrating and took far longer than expected.",
        "I've reported this same issue three times and nothing has been fixed.",
        "The recent redesign made the app slower and less intuitive overall.",
        "Customer service took over a week to respond to my last request.",
        "I'm unhappy that essential features are now locked behind a higher-priced plan.",
        "The product documentation is outdated and doesn't match the current version.",
        "I expected much better reliability given how much this plan costs.",
    ],
    "General Inquiry": [
        "What are your customer support hours and how can I reach a live agent?",
        "Do you offer discounts for students or non-profit organizations?",
        "Is there a mobile app available for iOS as well as Android?",
        "Can you tell me more about your data privacy and security practices?",
        "What integrations are currently supported with third-party tools?",
        "Do you have an enterprise plan with custom pricing for large teams?",
        "Where can I find your API documentation and getting started guide?",
        "Is training or onboarding support available for new team members?",
        "What is the difference between your Pro and Business plans?",
        "Can you point me to your public status page for outage updates?",
        "Do you have a referral program for existing customers?",
        "How long do you typically retain account data after cancellation?",
    ],
}

rows = []
for tag, tickets in CATEGORIES.items():
    for text in tickets:
        rows.append({"ticket_text": text, "true_tag": tag})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

# Also save to disk so it's easy to download/inspect as a standalone CSV artifact
df.to_csv("support_tickets.csv", index=False)

print(f"Total tickets: {len(df)}")
print(f"Categories: {df['true_tag'].nunique()}")
df.head(10)


In [ ]:
def clean_text(text: str) -> str:
    """Light preprocessing: strip whitespace, normalize spacing, remove stray control chars.
    We deliberately do NOT lowercase or strip punctuation aggressively, since LLMs
    perform better with natural, well-formed sentences rather than heavily
    normalized text (unlike classic bag-of-words models).
    """
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["ticket_text_clean"] = df["ticket_text"].apply(clean_text)

# Sanity check: no missing values, no empty strings
assert df["ticket_text_clean"].str.len().min() > 0
assert df["true_tag"].isna().sum() == 0

CANDIDATE_TAGS = sorted(df["true_tag"].unique().tolist())
print("Candidate tags:", CANDIDATE_TAGS)


### Dataset Exploration

In [ ]:
plt.figure(figsize=(9, 5))
order = df["true_tag"].value_counts().index
sns.countplot(data=df, y="true_tag", order=order, hue="true_tag", palette="viridis", legend=False)
plt.title("Support Ticket Distribution by Category")
plt.xlabel("Number of Tickets")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig("outputs_tag_distribution.png", dpi=150)
plt.show()


In [ ]:
# Train/test split: we use a small held-out "few-shot example pool" (2 examples
# per category, used only as in-context examples for prompting) and evaluate
# BOTH zero-shot and few-shot models on the remaining tickets, so the comparison
# is fair and neither approach is trained/tuned on the evaluation set.

FEWSHOT_EXAMPLES_PER_CLASS = 2

# Note: we use groupby(...).sample(...) rather than groupby(...).apply(lambda x: x.sample(...))
# because newer pandas versions can silently drop the grouping column from the result
# when using .apply() this way. .sample() is the safe, version-stable way to do a
# stratified per-category sample and preserves every column.
fewshot_pool = (
    df.groupby("true_tag", group_keys=False)
      .sample(n=FEWSHOT_EXAMPLES_PER_CLASS, random_state=42)
)
eval_df = df.drop(index=fewshot_pool.index).reset_index(drop=True)
fewshot_pool = fewshot_pool.reset_index(drop=True)

# Sanity check: fewshot_pool must retain the true_tag column and have the expected size
assert "true_tag" in fewshot_pool.columns, "true_tag column missing from fewshot_pool!"
assert len(fewshot_pool) == FEWSHOT_EXAMPLES_PER_CLASS * len(CANDIDATE_TAGS)

print(f"Few-shot example pool: {len(fewshot_pool)} tickets "
      f"({FEWSHOT_EXAMPLES_PER_CLASS} per category)")
print(f"Evaluation set: {len(eval_df)} tickets")


## 4. Zero-Shot Classification

We use `facebook/bart-large-mnli` through Hugging Face's `zero-shot-classification`
pipeline. This model was trained on Natural Language Inference (NLI) and reframes
classification as: *"does this ticket text entail the hypothesis 'This example is
about {tag}'?"* — no labeled training examples of our own are required, only the
list of candidate tag names.

The pipeline returns a probability for **every** candidate tag; we keep the
**top 3** as the ticket's predicted tags.


In [ ]:
from transformers import pipeline

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

def zero_shot_predict(text: str, candidate_tags=CANDIDATE_TAGS, top_k: int = 3):
    """Return the top-k predicted tags (highest probability first) for one ticket."""
    result = zero_shot_classifier(text, candidate_labels=candidate_tags, multi_label=True)
    ranked_tags = result["labels"][:top_k]
    ranked_scores = result["scores"][:top_k]
    return ranked_tags, ranked_scores

# Quick smoke test on a single ticket
sample_text = eval_df.iloc[0]["ticket_text_clean"]
tags, scores = zero_shot_predict(sample_text)
print("Ticket:", sample_text)
print("Top-3 zero-shot tags:", list(zip(tags, [round(s, 3) for s in scores])))


In [ ]:
# Run zero-shot classification over the full evaluation set
zero_shot_results = []

start = time.time()
for _, row in eval_df.iterrows():
    tags, scores = zero_shot_predict(row["ticket_text_clean"])
    zero_shot_results.append({
        "ticket_text": row["ticket_text"],
        "true_tag": row["true_tag"],
        "top3_tags": tags,
        "top3_scores": scores,
        "predicted_tag": tags[0],  # top-1 prediction for standard accuracy/F1
    })
elapsed = time.time() - start

zero_shot_df = pd.DataFrame(zero_shot_results)
print(f"Zero-shot inference completed on {len(zero_shot_df)} tickets in {elapsed:.1f}s")
zero_shot_df.head()


## 5. Few-Shot Prompting Classification

Here we use `google/flan-t5-base`, an instruction-tuned generative LLM, and give
it a **prompt containing labeled examples** (few-shot / in-context learning)
drawn from `fewshot_pool`, followed by the new ticket to classify. The model is
instructed to output the top 3 most likely tags, ranked, as a comma-separated
list — this mirrors how you would prompt a larger commercial LLM (e.g. GPT-4,
Claude) via an API; `flan-t5-base` is used here so the whole notebook runs for
free without requiring an API key.

> **Swapping in a commercial LLM API:** replace the body of `few_shot_predict()`
> with a call to your provider of choice (OpenAI, Anthropic, etc.), keeping the
> same prompt-construction logic — the rest of the notebook (evaluation,
> comparison, plots) works unchanged.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# NOTE: we load the tokenizer/model directly (instead of using pipeline("text2text-generation", ...))
# because some recent transformers releases have changed/renamed pipeline task aliases,
# which can raise a `KeyError: Unknown task text2text-generation` on certain versions.
# Loading the model directly avoids that entirely and works across transformers versions.

FEWSHOT_MODEL_NAME = "google/flan-t5-base"

few_shot_tokenizer = AutoTokenizer.from_pretrained(FEWSHOT_MODEL_NAME)
few_shot_model = AutoModelForSeq2SeqLM.from_pretrained(FEWSHOT_MODEL_NAME)

_device = "cuda" if torch.cuda.is_available() else "cpu"
few_shot_model = few_shot_model.to(_device)
print(f"Few-shot model loaded on: {_device}")


def generate_few_shot(prompt: str, max_new_tokens: int = 40) -> str:
    inputs = few_shot_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(_device)
    with torch.no_grad():
        output_ids = few_shot_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return few_shot_tokenizer.decode(output_ids[0], skip_special_tokens=True)


In [ ]:
def build_few_shot_prompt(ticket_text: str, examples: pd.DataFrame, tags=CANDIDATE_TAGS) -> str:
    """Construct an in-context-learning prompt with labeled examples."""
    instructions = (
        "You are a support-ticket tagging assistant. "
        f"Choose tags ONLY from this list: {', '.join(tags)}. "
        "Given a support ticket, respond with the 3 most relevant tags, "
        "ranked from most to least relevant, separated by commas. "
        "Do not explain your answer.\n\n"
    )
    example_block = ""
    for _, ex in examples.iterrows():
        example_block += f"Ticket: {ex['ticket_text_clean']}\nTags: {ex['true_tag']}\n\n"

    query_block = f"Ticket: {ticket_text}\nTags:"
    return instructions + example_block + query_block


def parse_tags_from_generation(raw_output: str, candidate_tags=CANDIDATE_TAGS, top_k: int = 3):
    """Parse the model's free-text output into a clean, ranked list of valid tags.
    Falls back gracefully: if the model outputs fewer than top_k valid tags,
    remaining slots are filled with the next-closest tag names by simple
    substring/fuzzy matching, so downstream evaluation never crashes on malformed output.
    """
    raw_candidates = [t.strip() for t in raw_output.split(",") if t.strip()]

    matched = []
    for cand in raw_candidates:
        # exact match first
        exact = [t for t in candidate_tags if t.lower() == cand.lower()]
        if exact:
            matched.append(exact[0])
            continue
        # fallback: substring match (handles minor generation noise)
        partial = [t for t in candidate_tags if cand.lower() in t.lower() or t.lower() in cand.lower()]
        if partial:
            matched.append(partial[0])

    # de-duplicate while preserving order
    seen = set()
    ranked = [t for t in matched if not (t in seen or seen.add(t))]

    # pad with remaining unused tags (in fixed order) if fewer than top_k were parsed
    for t in candidate_tags:
        if len(ranked) >= top_k:
            break
        if t not in ranked:
            ranked.append(t)

    return ranked[:top_k]


def few_shot_predict(text: str, examples: pd.DataFrame, top_k: int = 3):
    prompt = build_few_shot_prompt(text, examples)
    raw = generate_few_shot(prompt)
    tags = parse_tags_from_generation(raw, top_k=top_k)
    return tags, raw

# Quick smoke test
tags, raw = few_shot_predict(sample_text, fewshot_pool)
print("Ticket:", sample_text)
print("Raw model output:", raw)
print("Parsed top-3 few-shot tags:", tags)


In [ ]:
# Run few-shot classification over the full evaluation set
few_shot_results = []

start = time.time()
for _, row in eval_df.iterrows():
    tags, raw = few_shot_predict(row["ticket_text_clean"], fewshot_pool)
    few_shot_results.append({
        "ticket_text": row["ticket_text"],
        "true_tag": row["true_tag"],
        "top3_tags": tags,
        "raw_output": raw,
        "predicted_tag": tags[0],
    })
elapsed = time.time() - start

few_shot_df = pd.DataFrame(few_shot_results)
print(f"Few-shot inference completed on {len(few_shot_df)} tickets in {elapsed:.1f}s")
few_shot_df.head()


## 6. Evaluation: Zero-Shot vs. Few-Shot

We report three complementary metrics for each approach:

- **Top-1 Accuracy** — was the single most-confident predicted tag correct?
- **Macro-averaged Precision / Recall / F1** — treats every category equally
  regardless of class frequency, which matters since real ticket volume is
  rarely perfectly balanced across categories.
- **Top-3 Accuracy** — was the true tag *somewhere* in the top 3 predicted tags?
  This reflects the actual product use-case (a ranked shortlist of tags shown
  to a support agent), not just a single forced label.


In [ ]:
def top3_accuracy(results_df: pd.DataFrame) -> float:
    hits = results_df.apply(lambda r: r["true_tag"] in r["top3_tags"], axis=1)
    return hits.mean()


def evaluate(results_df: pd.DataFrame, name: str) -> dict:
    y_true = results_df["true_tag"]
    y_pred = results_df["predicted_tag"]

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    top3_acc = top3_accuracy(results_df)

    print(f"--- {name} ---")
    print(f"Top-1 Accuracy      : {acc:.3f}")
    print(f"Macro Precision     : {precision:.3f}")
    print(f"Macro Recall        : {recall:.3f}")
    print(f"Macro F1-score      : {f1:.3f}")
    print(f"Top-3 Accuracy      : {top3_acc:.3f}")
    print()

    return {
        "name": name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "top3_accuracy": top3_acc,
    }


zero_shot_metrics = evaluate(zero_shot_df, "Zero-Shot (BART-MNLI)")
few_shot_metrics = evaluate(few_shot_df, "Few-Shot (FLAN-T5, in-context examples)")

metrics_df = pd.DataFrame([zero_shot_metrics, few_shot_metrics]).set_index("name")
metrics_df


In [ ]:
print("Classification Report — Zero-Shot\n")
print(classification_report(zero_shot_df["true_tag"], zero_shot_df["predicted_tag"], zero_division=0))

print("Classification Report — Few-Shot\n")
print(classification_report(few_shot_df["true_tag"], few_shot_df["predicted_tag"], zero_division=0))


### Visualization: Zero-Shot vs. Few-Shot Comparison

In [ ]:
metrics_to_plot = ["accuracy", "precision", "recall", "f1", "top3_accuracy"]
plot_df = metrics_df[metrics_to_plot].reset_index().melt(
    id_vars="name", var_name="metric", value_name="score"
)

plt.figure(figsize=(10, 5.5))
sns.barplot(data=plot_df, x="metric", y="score", hue="name", palette="Set2")
plt.title("Zero-Shot vs. Few-Shot: Performance Comparison")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.xlabel("Metric")
plt.legend(title="")
plt.tight_layout()
plt.savefig("outputs_metric_comparison.png", dpi=150)
plt.show()


### Visualization: Confusion Matrix (Few-Shot, top-1 predictions)

In [ ]:
cm = confusion_matrix(few_shot_df["true_tag"], few_shot_df["predicted_tag"], labels=CANDIDATE_TAGS)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CANDIDATE_TAGS, yticklabels=CANDIDATE_TAGS,
)
plt.title("Confusion Matrix — Few-Shot Top-1 Predictions")
plt.xlabel("Predicted Tag")
plt.ylabel("True Tag")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("outputs_confusion_matrix.png", dpi=150)
plt.show()


## 7. Sample Predictions — Top 3 Tags per Ticket

Below is the final production-style output format: for each ticket, the system
returns the top 3 most probable tags (ranked), which is what would be surfaced
to a support agent for quick triage.


In [ ]:
def show_sample_predictions(results_df: pd.DataFrame, n: int = 8):
    sample = results_df.sample(n=n, random_state=7)
    for _, row in sample.iterrows():
        print(f"Ticket   : {row['ticket_text']}")
        print(f"True tag : {row['true_tag']}")
        print(f"Top-3    : {row['top3_tags']}")
        print("-" * 90)

print("### Few-Shot model sample predictions ###\n")
show_sample_predictions(few_shot_df)


In [ ]:
# Save full prediction outputs for the submission / README results table
zero_shot_df.to_csv("outputs_zero_shot_predictions.csv", index=False)
few_shot_df.to_csv("outputs_few_shot_predictions.csv", index=False)
metrics_df.to_csv("outputs_metrics_summary.csv")

print("Saved: outputs_zero_shot_predictions.csv, outputs_few_shot_predictions.csv, outputs_metrics_summary.csv")


## 8. Final Summary & Insights

**Approach.** Two LLM-based strategies were built and compared for auto-tagging
free-text support tickets across 8 categories: (1) zero-shot classification with
`facebook/bart-large-mnli` using natural language inference, and (2) few-shot
prompting with `google/flan-t5-base` using 2 labeled in-context examples per
category.

**Key observations** *(fill in with the actual numbers printed above once you
run the notebook — this section is written for you to complete after execution,
since results depend on the exact model versions and random seed at run time)*:

- Few-shot prompting is expected to outperform zero-shot on **Top-1 accuracy and
  macro F1**, because the labeled examples give the model a concrete pattern for
  both the *decision boundary* between similar categories (e.g. "Bug Report" vs.
  "Technical Issue") and the *expected output format*.
- Zero-shot classification tends to do reasonably well on categories with very
  distinctive vocabulary (e.g. "Billing & Payment", "Cancellation & Refund") but
  struggles more on categories that overlap semantically (e.g. distinguishing
  "Bug Report" from "Technical Issue", or "Product Complaint" from "General
  Inquiry").
- **Top-3 accuracy** is meaningfully higher than Top-1 accuracy for both
  approaches, which supports the product decision to surface a ranked shortlist
  of 3 tags to human agents rather than forcing a single automatic label.
- The confusion matrix highlights which category pairs are most frequently
  confused, which is directly actionable: those categories could be merged,
  more clearly defined, or given more few-shot examples in production.

**Limitations.**
- The evaluation set (96 hand-authored tickets) is small and synthetic; results
  on real, noisier production ticket data may differ.
- `flan-t5-base` is a relatively small generative model chosen for free,
  API-key-free reproducibility — swapping in a larger commercial LLM (GPT-4,
  Claude, etc.) via API would likely further improve few-shot performance,
  using the exact same prompt-engineering scaffold built in Section 5.

**Next steps for production.**
- Increase the few-shot example pool and use retrieval to dynamically select the
  most similar few-shot examples per incoming ticket (a lightweight form of RAG).
- Add a confidence threshold below which tickets are routed to a human for
  manual tagging instead of being auto-tagged.
- Continuously log agent corrections to the auto-assigned tags and use them to
  refine the few-shot example pool over time.


## 9. (Optional) Download Your Results from Colab

If you're running this in Google Colab and want to save the generated CSVs,
charts, and dataset to your own computer, run the cell below.


In [ ]:
try:
    from google.colab import files
    for fname in [
        "support_tickets.csv",
        "outputs_tag_distribution.png",
        "outputs_metric_comparison.png",
        "outputs_confusion_matrix.png",
        "outputs_zero_shot_predictions.csv",
        "outputs_few_shot_predictions.csv",
        "outputs_metrics_summary.csv",
    ]:
        try:
            files.download(fname)
        except FileNotFoundError:
            print(f"Skipping {fname} (not found — did all earlier cells run successfully?)")
except ImportError:
    print("Not running in Google Colab — files are already saved in the local working directory.")
